# 🎬 FilmBot — 1-Click Google Colab Runner

**100% Pre-configured with all 3 Google Drives (`gdrive`, `drive_tharusha`, `drive_sithum`), credentials & session!**

### 🚀 Run කරන විදිය:
උඩ Menu එකෙන් **Runtime → Run all** (හෝ `Ctrl + F9`) click කරන්න.
ඔයාට කිසිම දෙයක් type කරන්නවත් secrets දාන්නවත් ඕන නෑ. සේරම auto setup වෙලා bot එක start වෙනවා!

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies                               ║
# ╚══════════════════════════════════════════════════════════════╝
print('📦 1/5 Installing system tools (ffmpeg, aria2, rclone)...')
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 rclone git curl wget unzip

print('📦 2/5 Installing Python libraries...')
!pip install --upgrade pyrogram tgcrypto httpx python-dotenv aiofiles Pillow beautifulsoup4 lxml pikpakapi gdown fastapi 'uvicorn[standard]' imageio-ffmpeg nest_asyncio

import pyrogram, nest_asyncio
nest_asyncio.apply()
print(f'✅ All tools installed! Pyrogram v{pyrogram.__version__}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Clone / Update Bot Repo                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os
BOT_DIR = '/content/film_web_site'
REPO_URL = 'https://github.com/lakindugimsara50-del/film-bot.git'

if os.path.exists(f'{BOT_DIR}/.git'):
    print('🔄 Repo already exists — pulling latest changes...')
    !git -C {BOT_DIR} pull origin main
else:
    print('📥 Cloning FilmBot from GitHub...')
    !git clone {REPO_URL} {BOT_DIR}

assert os.path.exists(f'{BOT_DIR}/bot/main.py'), '❌ Clone failed!'
print('✅ Bot codebase ready!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Auto-Configure 3 Drives, .env & Telegram Session   ║
# ╚══════════════════════════════════════════════════════════════╝
import base64, os, json

BOT_PATH = '/content/film_web_site/bot'
DATA_DIR = f'{BOT_PATH}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# 1. Unpack rclone.conf (All 3 Drives: gdrive, drive_tharusha, drive_sithum)
RCLONE_B64 = 'W2dkcml2ZV0NCnR5cGUgPSBkcml2ZQ0Kc2NvcGUgPSBkcml2ZQ0KdG9rZW4gPSB7ImFjY2Vzc190b2tlbiI6InlhMjkuYTBBWDA3Q21zeXMxVzJVb3RYbUxneE8wWEZmTEl4aUNhNUtWZmZUMzIwRzZHeFhVazNBMk9NaW01MC1HMlFaY3lOdUlVNEVPdW9TRFBTVzR0SU10RDBGR19VaVlLRzk0cVllUFBHTkRSUnlLRTRveW1NdTVpLS1UMm9wblUxOVJ6a1JPOUI1TmJkZ3ItZFFDMXRoUHhCRlhhTEQyQUNNcE82cnN6RjczVzhjTk01WkZ5WjhnM3RsWk5tTU1pR1FfYzIzbDhLcDFjYUNnWUtBVFVTQVJZU0ZRSEdYMk1pVk5hQ3hydDJJNF9KZWNjWE1SWEZvdzAyMDYiLCJ0b2tlbl90eXBlIjoiQmVhcmVyIiwicmVmcmVzaF90b2tlbiI6IjEvLzBna2lMZnd4RGFQaWZDZ1lJQVJBQUdCQVNOd0YtTDlJclRYVVk5OUJOTm5kaWZjN1FITmlMMmhVbDB6cHFoSzg5cVNBWkRRcE1vd0dNdWhmcS1uSVdkWjc2RkNaWGxZdnk5VFEiLCJleHBpcnkiOiIyMDI2LTA5LTIzVDE0OjEwOjU1LjczOTE3MDgrMDU6MzAiLCJleHBpcmVzX2luIjozNTk5fQ0KdGVhbV9kcml2ZSA9IA0KDQpbZHJpdmVfc2l0aHVtXQ0KdHlwZSA9IGRyaXZlDQpzY29wZSA9IGRyaXZlDQp0b2tlbiA9IHsiYWNjZXNzX3Rva2VuIjoieWEyOS5hMEFYMDdDbXN2YngyREFOc2MyWk1wVnJYZ0Etc05HNkkyVXFObEFkYk84UEJWRTRPdV9IdlNNUmkzc3JldVdVNXBYYzZFSkNMZXZ4MlNiZTZKZmdBYzRKVEFjaDFncWxnSlhHZ3BKbFcyY2ZHdC14U1pzN2thdlhtWTktM3lSMlduT3B5OUpNcEJFd1Fxb282SkYtamZvYXUtVEljeTRRODBjeGdHclJCQWlqOS1tTER6Q1hiWkZOMkdzTE5fT1Z0S2wtLUZoNkFhQ2dZS0FmZ1NBUklTRlFIR1gyTWk5dkkwUFZ6aTJZNWRSU3JkWUpEdm1RMDIwNiIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJyZWZyZXNoX3Rva2VuIjoiMS8vMGdPV1U4LVcxa0RUWkNnWUlBUkFBR0JBU053Ri1MOUlyOXFpR1Z3ODdOTmljMlQ1NWNrdkUwaG83OG9uaTRWWXhrMVVsYTV4c1pHZkpUZWM3N0ZwNUZnSFFOY1B3QjNtWTJ0ZyIsImV4cGlyeSI6IjIwMjYtMDktMjNUMTY6MTE6MzguMDk1MDA4KzA1OjMwIiwiZXhwaXJlc19pbiI6MzU5OX0NCnRlYW1fZHJpdmUgPSANCg0KW2RyaXZlX3RoYXJ1c2hhXQ0KdHlwZSA9IGRyaXZlDQpzY29wZSA9IGRyaXZlDQp0b2tlbiA9IHsiYWNjZXNzX3Rva2VuIjoieWEyOS5hMEFkTUQ2RWpyNWtTSXNWQnZmUnN5RG5lOVF1em5UQ1RvS3VnYWdhRDZfY0daeE1nVV9GX01ESVJIcXg4aFBwUFZUVTRkRDhGa0NVYUYwRm5mYnhZNXQ1VkVpUWNRZkNRem5KWGc2M3lCUjRKQnQyQnM0QXN5Vk5HTnFYQzF1ejJLOE9hdWotbmdEZVRxYkJFbHRMdUhiN0lpNnhXX05mOExVSDE3OHdLRWlaaXdtT0Qwa2VTMXpLazdCcWJ1QXYxTnlZWTgtZWdhQ2dZS0FRc1NBUklTRlFIR1gyTWlzZVB0QlZlSkxDcTg1VkEtUjQ0VUxBMDIwNiIsInRva2VuX3R5cGUiOiJCZWFyZXIiLCJyZWZyZXNoX3Rva2VuIjoiMS8vMGdUNmxtT2l3YjVfZUNnWUlBUkFBR0JBU053Ri1MOUlyQlgzNGVQeU9UWnc5bXJZRk1VNTh2bjJQcmYyMkdGbDJhU0xsVG5rdzFLNGRYU3RVNE1HeUNCOHpqbjJrbEpuaVRsRSIsImV4cGlyeSI6IjIwMjYtMDktMjNUMTY6MzM6MTEuMDk0OTA0OSswNTozMCIsImV4cGlyZXNfaW4iOjM1OTl9DQp0ZWFtX2RyaXZlID0gDQoNCg=='
with open(f'{DATA_DIR}/rclone.conf', 'wb') as f:
    f.write(base64.b64decode(RCLONE_B64))
print('✅ rclone.conf ready with all 3 Drives!')

# 2. Unpack .env
ENV_B64 = 'VEdfQVBJX0lEPTM5MjU0ODg1DQpUR19BUElfSEFTSD1kOTJhNWZkZDE5YThiYzY5ZWU3M2VlOTFjYWUwOWU5OA0KQk9UX1RPS0VOPTg5MzAxNjc1MDM6QUFIcWRXdGhIUGdrVFp0Qld5aTFCcFRKU3BXaVZ1Y2hnRE0NCkFETUlOX0lEUz04NzYyMzI5MjI2DQpQUklWQVRFX0NIQU5ORUxfSUQ9LTEwMDQzMjU3NTk1MDUNClBVQkxJQ19DSEFOTkVMX0lEPS0xMDA0MzI1NzU5NTA1DQpUTURCX0FQSV9LRVk9ZGNiZTMyNjJjOTAxNzE5ZWQ4YjAyOTFlNzUxYmM2OWENCkdJVEhVQl9UT0tFTj1naHBfc2ZZR3MyOFg5TVg2T3lkbXQ4eDNkNUdQd01CNGtSM2NUT2JDDQpHSVRIVUJfUkVQTz1sYWtpbmR1Z2ltc2FyYTUwLWRlbC9maWxtLWJvdA0KU1RSRUFNX0JBU0VfVVJMPWh0dHBzOi8vc3RyZWFtLnlvdXJkb21haW4ud29ya2Vycy5kZXYNClNJVEVfQkFTRV9VUkw9aHR0cHM6Ly9maWxtc3ViLnBhZ2VzLmRldg0KU0VTU0lPTl9OQU1FPXNlc3Npb24NClNFRURSX1VTRVJOQU1FPWxha2luZHVnaW1zYXJhNTBAZ21haWwuY29tDQpTRUVEUl9QQVNTV09SRD1JbmRpa2FAMjEyMjEzOA0K'
with open(f'{BOT_PATH}/.env', 'wb') as f:
    f.write(base64.b64decode(ENV_B64))
print('✅ .env configuration ready!')

# 3. Unpack drives.json so drive_manager manages all 3 drives
drives_json = [
    {'name': 'gdrive',         'type': 'rclone', 'rclone_remote': 'gdrive',         'enabled': True},
    {'name': 'drive_tharusha', 'type': 'rclone', 'rclone_remote': 'drive_tharusha', 'enabled': True},
    {'name': 'drive_sithum',   'type': 'rclone', 'rclone_remote': 'drive_sithum',   'enabled': True}
]
with open(f'{DATA_DIR}/drives.json', 'w') as f:
    json.dump(drives_json, f, indent=2)
print('✅ drives.json registered 3 Google Drives!')

# 4. Unpack Telegram session (no phone OTP prompts!)
SESSION_B64 = '...'
if SESSION_B64:
    with open(f'{BOT_PATH}/film_bot.session', 'wb') as f:
        f.write(base64.b64decode(SESSION_B64))
    with open(f'{BOT_PATH}/session.session', 'wb') as f:
        f.write(base64.b64decode(SESSION_B64))
    print('✅ Telegram session restored (no login prompt required)!')

# 5. Check Drive connections
print('\n🔍 Verifying all 3 Google Drives...')
for drv in ['gdrive', 'drive_tharusha', 'drive_sithum']:
    res = !rclone lsd {drv}: --config {DATA_DIR}/rclone.conf --max-depth 1 2>&1 | head -2
    has_err = any('ERROR' in l or 'Failed' in l for l in res)
    st = '⚠️ ' if has_err else '✅ Connected:'
    print(f'   {st} {drv}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Patch Size Limits for Colab (4GB Files)            ║
# ╚══════════════════════════════════════════════════════════════╝
import re, os
BOT_PATH = '/content/film_web_site/bot'

def patch(path, pattern, replacement, desc):
    if os.path.exists(path):
        txt = open(path).read()
        new = re.sub(pattern, replacement, txt)
        if new != txt:
            open(path, 'w').write(new)
            print(f'  ✅ {desc}')
        else:
            print(f'  ℹ️ {desc} (already applied)')

print('🔧 Applying Colab 4GB limits patch (100GB Disk)...')
patch(f'{BOT_PATH}/services/scrapers/torrent_finder.py',
      r'MAX_FILE_SIZE_BYTES\s*=\s*int\([^)]+\)',
      'MAX_FILE_SIZE_BYTES = int(4 * 1024 * 1024 * 1024)',
      'torrent_finder: MAX_FILE_SIZE_BYTES -> 4GB')

patch(f'{BOT_PATH}/services/downloader.py',
      r'1\.85\s*\*\s*1024\s*\*\s*1024\s*\*\s*1024',
      '4 * 1024 * 1024 * 1024',
      'downloader: 1.85GB watchdog -> 4GB')

patch(f'{BOT_PATH}/services/leech_service.py',
      r'1\.85\s*\*\s*1024\s*\*\s*1024\s*\*\s*1024',
      '4 * 1024 * 1024 * 1024',
      'leech_service: 1.85GB preflight -> 4GB')
print('✅ Patch applied!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — 🚀 START THE BOT                                  ║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, nest_asyncio
nest_asyncio.apply()

BOT_DIR = '/content/film_web_site/bot'
if BOT_DIR not in sys.path:
    sys.path.insert(0, BOT_DIR)

os.chdir(BOT_DIR)
print(f'📂 Working dir: {os.getcwd()}')
print('🤖 Starting @Filmsinhala200Bot...')
print('─' * 55)
print('📌 Bot is live! Press ■ STOP button in Colab to stop.')
print('─' * 55)

%run main.py
